## Imports & Parameters

In [1]:
## Import
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
local_extract = cwd + '/local_extract/'

print(cwd)
print(local_extract)

/Users/E2074/analytics/customer/Appography/age_classification_notebook
/Users/E2074/analytics/customer/Appography/age_classification_notebook/local_extract/


In [9]:
## Parameter
start_date = '20240701'
end_date = '20240710'
refresh_month = 'july'
city = 'Bangalore'

## Datasets

### Metric data

print('Query 1 running..')
start_time = time.time()

active_customer_base = f"""

    WITH fe_tbl AS (
    
        SELECT 
            yyyymmdd,
            platform,
            service_detail_id,
            user_id customer_id,
            fare_estimate_id,
            quarter_hour fe_quarter_hour
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        ),
        
        rr_net_tbl as (
        
        SELECT   
            yyyymmdd,
            service_detail_id,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            quarter_hour rr_quarter_hour
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city_name = '{city}'
            AND service_obj_service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
        ),
        
        merged AS (
        
        SELECT
            fe_tbl.*,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.rr_quarter_hour
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
            AND fe_tbl.customer_id = rr_net_tbl.customer_id
            AND fe_tbl.service_detail_id = rr_net_tbl.service_detail_id
        )

        SELECT
            platform,
            customer_id,
            COUNT(DISTINCT fare_estimate_id) AS fe_count,
            COUNT(DISTINCT order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN fare_estimate_id END) morning_fe,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN fare_estimate_id END) evening_fe,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN order_id END) morning_rr,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN order_id END) evening_rr

        FROM 
            merged
        GROUP BY 1,2
        
"""

df_active_customer_base = pd.read_sql(active_customer_base, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_active_customer_base.head(1)
df_active_customer_base.to_csv(local_extract + 'raw_active_customer_base_{}_{}.csv'.format(refresh_month,city), index=False)

In [11]:
df_active_customer_base = pd.read_csv(local_extract + 'raw_active_customer_base_{}_{}.csv'.format(refresh_month,city))

### Segment data

print('Query 2 running..')
start_time = time.time()
customer_segments = f"""

    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_service_affinity service_affinity
        
    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date =  DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d'), '%Y-%m-%d') -- + INTERVAL '1' DAY 
        AND taxi_recency_segment != 'INACTIVE' 
        AND taxi_lifetime_last_ride_city = '{city}'
        
"""
df_customer_segments = pd.read_sql(customer_segments, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_segments.to_csv(local_extract + 'raw_customer_segments_{}_{}.csv'.format(refresh_month,city), index=False)

In [16]:
df_customer_segments = pd.read_csv(local_extract + 'raw_customer_segments_{}_{}.csv'.format(refresh_month,city))

### Age and gender data

print('Query 3 running..')
start_time = time.time()

customer_demo_data = f"""

    WITH fe_customer AS (

        SELECT 
            user_id customer_id
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId customer_id,
            CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            a.customer_id,
            COALESCE(age, 0) age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            fe_customer a
        LEFT JOIN 
            demographic_data b
            on a.customer_id = b.customer_id        
"""

df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_demo_data.to_csv(local_extract + 'raw_customer_demo_data_{}_{}.csv'.format(refresh_month,city), index=False)

In [18]:
df_customer_demo_data = pd.read_csv(local_extract + 'raw_customer_demo_data_{}_{}.csv'.format(refresh_month,city))

### Use-case data

print('Query 4 running..')
start_time = time.time()

customer_use_case_data = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            WHERE 
                aoi = 'Bangalore, India'
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
            AND city_name = '{city}'
            
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21
            AND city_name = '{city}'
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        ) 
"""

df_customer_use_case_data = pd.read_sql(customer_use_case_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_use_case_data.to_csv(local_extract + 'raw_customer_use_case_data_{}_{}.csv'.format(refresh_month,city), index=False)

In [20]:
df_customer_use_case_data = pd.read_csv(local_extract + 'raw_customer_use_case_data_{}_{}.csv'.format(refresh_month,city))

### Appography data

print('Query 5 running..')
start_time = time.time()
customer_appography_query = f"""

    WITH customer_app AS (

        SELECT
            yyyymmdd AS last_sync_date,
            id user_id, 
            CAST(json_parse(appography_data) AS ARRAY(VARCHAR)) AS app_lists,
            DENSE_RANK() over(partition by id order by yyyymmdd DESC) updated_seq
        FROM 
            customer.info_customer_installed_apps_immutable
        WHERE 
            yyyymmdd >= DATE_FORMAT(DATE_PARSE('{start_date}', '%Y%m%d') - INTERVAL '15' DAY  , '%Y%m%d')
            AND yyyymmdd <= '{end_date}'
            AND (city = '{city}' OR city IS NULL OR city = '')
        )
        
        SELECT
            user_id,
            app_lists
        FROM 
            customer_app
        WHERE 
            updated_seq = 1
        
"""
df_customer_appography_data = pd.read_sql(customer_appography_query, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_appography_data.to_csv(local_extract + 'raw_customer_appography_data_{}_{}.csv'.format(refresh_month,city), index=False)

In [27]:
df_customer_appography_data = pd.read_csv(local_extract + 'raw_customer_appography_data_{}_{}.csv'.format(refresh_month,city))

#Only selected city
df_combined_data_dumper = pd.merge(df_customer_demo_data['customer_id'], df_customer_appography_data,
                                   how='inner', 
                                   left_on=['customer_id'], right_on=['user_id'] 
                                  ) 

df_combined_data_dumper = df_combined_data_dumper[['customer_id', 'app_lists']]

#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_combined_data_dumper['app_lists'] = df_combined_data_dumper['app_lists'].apply(lambda x: extract_and_trim(x))
df_customer_appography_data = df_combined_data_dumper[['customer_id', 'app_lists']]

df_customer_appography_data.to_csv(local_extract + 'filtered_customer_appography_data_{}_{}.csv'.format(refresh_month,city), 
                                   index=False)

#delete df_combined_data_dumper
del df_combined_data_dumper

In [30]:
start_time = time.time()
df_customer_appography_data = pd.read_csv(local_extract + 'filtered_customer_appography_data_{}_{}.csv'.format(refresh_month,city))

# regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_appography_data['app_lists'] = df_customer_appography_data['app_lists'].apply(lambda x: extract_and_trim(x))

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

Execution time :  0.39  mins


### Merge

def merging_df(df1, df2, df3, df4, df5):

    start_time = time.time()

    df_active_customer_base = df1
    df_customer_segments = df2
    df_customer_demo_data = df3
    df_customer_use_case_data = df4
    df_customer_appography_data = df5

    print(df_active_customer_base.columns)
    print(df_customer_segments.columns)
    print(df_customer_demo_data.columns)
    print(df_customer_use_case_data.columns)
    print(df_customer_appography_data.columns)
    
    print('Merging 1.. ')
    merge1 = pd.merge(df_active_customer_base,df_customer_segments,how='left',on = ['customer_id'])
    
    print('Merging 2..')
    merge2 = pd.merge(merge1,df_customer_demo_data,how='left',on = ['customer_id'])
    
    print('Merging 3..')
    merge3 = pd.merge(merge2,df_customer_use_case_data,how='left',on = ['customer_id'])
    
    print('Merging 4..')
    merge4 = pd.merge(merge3,df_customer_appography_data,how='left',on = ['customer_id'])
    
    df_all_customer = merge4
    
    del merge1
    del merge2
    del merge3
    del merge4
    
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")

    df_all_customer.to_csv(local_extract + 'processed_all_customer_{}_{}.csv'.format(refresh_month,city), index=False)

    return df_all_customer

df_all_customer = merging_df(df_active_customer_base, 
                             df_customer_segments, 
                             df_customer_demo_data, 
                             df_customer_use_case_data, 
                             df_customer_appography_data)

In [36]:
df_all_customer = pd.read_csv(local_extract + 'processed_all_customer_{}_{}.csv'.format(refresh_month,city))

# regular-exp 
def extract_and_trim(text):
    if isinstance(text, str):
        return [val.strip() for val in re.findall(r"'([^']*)'", text)]
    else:
        return []

df_all_customer['app_lists'] = df_all_customer['app_lists'].apply(lambda x: extract_and_trim(str(x)))

In [37]:
df_all_customer.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_lists
0,android,62a1e8c317af9a37d451fd8a,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,0,MALE,Unknown,health_and_personal,educational,Unknown,"[Airtel Thanks, Dream11, Zomato, Dream11, Face..."


In [38]:
# Replace empty lists with NaN
df_all_customer['app_lists'] = df_all_customer['app_lists'].apply(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)

## Distribution check (Optional-Can-Skip)

In [39]:
## platform customer distribution
df_temp = df_all_customer.groupby(['platform']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,customers,% distribution
0,android,1518051,80.22
1,ios,374255,19.78


### Age customer distribution¶

In [40]:
df_all_customer['age_presence'] = np.where(df_all_customer['age'] > 0, 'AVAILABLE', 'NA')

df_temp = df_all_customer\
                .groupby(['age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,customers,% distribution
0,AVAILABLE,637350,33.71
1,NA,1253533,66.29


In [41]:
df_temp = df_all_customer\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,age_presence,customers,total,% distribution
0,android,AVAILABLE,493358,1518051,32.50
1,android,NA,1024693,1518051,67.50
2,ios,AVAILABLE,144612,374255,38.64
3,ios,NA,229643,374255,61.36


### Gender distribution

In [42]:
df_temp = df_all_customer\
                .groupby(['gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,gender,customers,% distribution
0,FEMALE,576356,30.48
1,MALE,1265044,66.90
2,OTHERS,6351,0.34
3,UNKNOWN,43132,2.28


In [43]:
df_temp = df_all_customer\
                .groupby(['platform', 'gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,gender,customers,total,% distribution
0,android,FEMALE,437296,1518051,28.81
1,android,MALE,1032884,1518051,68.04
2,android,OTHERS,5564,1518051,0.37
3,android,UNKNOWN,42307,1518051,2.79
4,ios,FEMALE,139520,374255,37.28
5,ios,MALE,233121,374255,62.29
6,ios,OTHERS,789,374255,0.21
7,ios,UNKNOWN,825,374255,0.22


### Appography distribution

In [44]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_lists'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,appo_presence,customers,% distribution
0,android,AVAILABLE,1395254,91.91
1,android,NA,122797,8.09


### Opportunity Size

In [45]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_lists'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,age_presence,appo_presence,customers,% distribution
0,android,AVAILABLE,AVAILABLE,450749,29.69
1,android,AVAILABLE,NA,42609,2.81
2,android,NA,AVAILABLE,944505,62.22
3,android,NA,NA,80188,5.28


In [46]:
df_temp.pivot(index ='age_presence' , columns ='appo_presence', values =['% distribution'])

% distribution      
appo_presence      AVAILABLE    NA
age_presence                      
AVAILABLE              29.69  2.81
NA                     62.22  5.28

## One-Hot-Encoder

In [47]:
categorical_columns = ['gender', 'ltr_segment', 'life_time_stage', 'service_affinity', 
                       'first_half_drop_primary', 'first_half_drop_secondary',
                       'second_half_pickup_primary', 'second_half_pickup_secondary']

def one_hot_encoder(df):
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_unknown')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)
    
    
    return df

In [48]:
df_one_hot_encoded = one_hot_encoder(df_all_customer)
print(df_one_hot_encoded.shape)
df_one_hot_encoded['life_time_rides'].fillna(0, inplace=True)
df_one_hot_encoded.head(3)

(1944738, 89)


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,life_time_rides,age,app_lists,age_presence,appo_presence,gender_FEMALE,gender_MALE,gender_OTHERS,gender_UNKNOWN,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,life_time_stage_DORMANT,life_time_stage_HANDHOLDING,life_time_stage_HOOK,life_time_stage_SOFT_CHURN,life_time_stage_SUSTENANCE,life_time_stage_UNKNOWN,life_time_stage_life_time_stage_unknown,service_affinity_AUTO_CAB,service_affinity_LINK_AUTO,service_affinity_LINK_CAB,service_affinity_NO_AFFINITY,service_affinity_ONLY_AUTO,service_affinity_ONLY_CAB,service_affinity_ONLY_LINK,service_affinity_UNKNOWN,service_affinity_service_affinity_unknown,first_half_drop_primary_Unknown,first_half_drop_primary_educational,first_half_drop_primary_first_half_drop_primary_unknown,first_half_drop_primary_food,first_half_drop_primary_govt_amenity,first_half_drop_primary_health_and_personal,first_half_drop_primary_household_needs,first_half_drop_primary_leisure,first_half_drop_primary_office,first_half_drop_primary_place_of_worship,first_half_drop_primary_residential,first_half_drop_primary_transit_station,first_half_drop_secondary_Unknown,first_half_drop_secondary_educational,first_half_drop_secondary_first_half_drop_secondary_unknown,first_half_drop_secondary_food,first_half_drop_secondary_govt_amenity,first_half_drop_secondary_health_and_personal,first_half_drop_secondary_household_needs,first_half_drop_secondary_leisure,first_half_drop_secondary_office,first_half_drop_secondary_place_of_worship,first_half_drop_secondary_residential,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,android,62a1e8c317af9a37d451fd8a,45,9,4,6,39,5,4,12.0,0,"[Airtel Thanks, Dream11, Zomato, Dream11, Face...",NA,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,android,620f9130d96a7ae295c40336,22,9,3,0,22,0,9,100.0,0,"[Slack, Airtel Thanks, Apollo 247, Zomato, Axi...",NA,AVAILABLE,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,android,617e60af544f1291db483b91,7,4,2,2,5,1,3,80.0,20,"[Adobe Scan, Amazon Prime Video, Amazon Music,...",AVAILABLE,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0

In [50]:
df_one_hot_encoded.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr',
       'life_time_rides', 'age', 'app_lists', 'age_presence', 'appo_presence',
       'gender_FEMALE', 'gender_MALE', 'gender_OTHERS', 'gender_UNKNOWN',
       'ltr_segment_HH', 'ltr_segment_PHH', 'ltr_segment_UNKNOWN',
       'ltr_segment_ltr_segment_unknown', 'life_time_stage_CHURN_OTB',
       'life_time_stage_COMMITTED', 'life_time_stage_DETOX',
       'life_time_stage_DORMANT', 'life_time_stage_HANDHOLDING',
       'life_time_stage_HOOK', 'life_time_stage_SOFT_CHURN',
       'life_time_stage_SUSTENANCE', 'life_time_stage_UNKNOWN',
       'life_time_stage_life_time_stage_unknown', 'service_affinity_AUTO_CAB',
       'service_affinity_LINK_AUTO', 'service_affinity_LINK_CAB',
       'service_affinity_NO_AFFINITY', 'service_affinity_ONLY_AUTO',
       'service_affinity_ONLY_CAB', 'service_affinity_ONLY_LINK',
       'service_affinity_UNKNOWN', 'service_

In [51]:
encoded_columns = ['customer_id', 
#                    'fe_count', 'rr_count', 'net_count',
#                    'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', # 'life_time_rides', 
#                    'gender_FEMALE', 'gender_MALE', 'gender_OTHERS','gender_UNKNOWN', 
                   'ltr_segment_HH', 'ltr_segment_PHH', 'ltr_segment_UNKNOWN', 
                   'life_time_stage_CHURN_OTB', 'life_time_stage_COMMITTED', 'life_time_stage_DETOX', 
                   'life_time_stage_DORMANT', 'life_time_stage_HANDHOLDING', 'life_time_stage_HOOK',
                   'life_time_stage_SOFT_CHURN', 'life_time_stage_SUSTENANCE', 'life_time_stage_UNKNOWN', 
                   'service_affinity_AUTO_CAB', 'service_affinity_LINK_AUTO','service_affinity_LINK_CAB', 
                   'service_affinity_NO_AFFINITY', 'service_affinity_ONLY_AUTO', 'service_affinity_ONLY_CAB',
                   'service_affinity_ONLY_LINK', 'service_affinity_UNKNOWN',
                   'first_half_drop_primary_Unknown', 'first_half_drop_primary_educational',
                   'first_half_drop_primary_first_half_drop_primary_unknown', 'first_half_drop_primary_food', 
                   'first_half_drop_primary_govt_amenity',
                   'first_half_drop_primary_health_and_personal', 'first_half_drop_primary_household_needs',
                   'first_half_drop_primary_leisure', 'first_half_drop_primary_office',
                   'first_half_drop_primary_place_of_worship', 'first_half_drop_primary_residential',
                   'first_half_drop_primary_transit_station', 'first_half_drop_secondary_Unknown',
                   'first_half_drop_secondary_educational', 'first_half_drop_secondary_first_half_drop_secondary_unknown',
                   'first_half_drop_secondary_food', 'first_half_drop_secondary_govt_amenity',
                   'first_half_drop_secondary_health_and_personal', 'first_half_drop_secondary_household_needs',
                   'first_half_drop_secondary_leisure', 'first_half_drop_secondary_office', 'first_half_drop_secondary_place_of_worship',
                   'first_half_drop_secondary_residential', 'first_half_drop_secondary_transit_station',
                   'second_half_pickup_primary_Unknown', 'second_half_pickup_primary_educational',
                   'second_half_pickup_primary_food', 'second_half_pickup_primary_govt_amenity',
                   'second_half_pickup_primary_health_and_personal', 'second_half_pickup_primary_household_needs',
                   'second_half_pickup_primary_leisure', 'second_half_pickup_primary_office',
                   'second_half_pickup_primary_place_of_worship', 'second_half_pickup_primary_residential',
                   'second_half_pickup_primary_second_half_pickup_primary_unknown', 'second_half_pickup_primary_transit_station',
                   'second_half_pickup_secondary_Unknown', 'second_half_pickup_secondary_educational',
                   'second_half_pickup_secondary_food', 'second_half_pickup_secondary_govt_amenity',
                   'second_half_pickup_secondary_health_and_personal', 'second_half_pickup_secondary_household_needs',
                   'second_half_pickup_secondary_leisure', 'second_half_pickup_secondary_office',
                   'second_half_pickup_secondary_place_of_worship', 'second_half_pickup_secondary_residential',
                   'second_half_pickup_secondary_second_half_pickup_secondary_unknown', 'second_half_pickup_secondary_transit_station']

df_one_hot_encoded = df_one_hot_encoded[encoded_columns]

## Appography & Use-Case

In [52]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_lists', 'age_presence', 'appo_presence'],
      dtype='object')

In [54]:
df_triangulation_1 = df_all_customer[(df_all_customer['platform'].isin(['android']) 
                                      & 
                                      df_all_customer['appo_presence'].isin(['AVAILABLE']))]\
                                    [['customer_id', 'age', 'appo_presence','app_lists',
                                      'fe_count', 'rr_count', 'net_count',
                                      'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 
                                      'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity',
                                      'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary',
                                     ]]

df_triangulation_1.head(2)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,"[Airtel Thanks, Dream11, Zomato, Dream11, Face...",45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown
1,620f9130d96a7ae295c40336,0,AVAILABLE,"[Slack, Airtel Thanks, Apollo 247, Zomato, Axi...",22,9,3,0,22,0,9,PHH,100.0,COMMITTED,LINK_AUTO,FEMALE,NaN,NaN,office,transit_station


### Explode

In [55]:
## Explode list into individual rows
df_main_apps_explode = df_triangulation_1.explode('app_lists')
df_main_apps_explode['app_lists'] = df_main_apps_explode['app_lists'].str.lower()

df_main_apps_explode.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,airtel thanks,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,zomato,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown


### Mobile app to category mapping

In [56]:
category_list = pd.read_csv('category_mapping.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['category'] = category_list['category'].str.lower()
category_list = category_list[['app_name', 'category']]
print(category_list.shape)
print(category_list.category.unique())
category_list.head(1)

(538, 2)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app' 'home_service' 'insurance' 'job_search' 'vehicle'
 'government']


,app_name,category
0,ignou e-content,educational


### Merge

In [57]:
df_main_apps_explode.head(2)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,airtel thanks,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown


In [58]:
category_list.head(2)

,app_name,category
0,ignou e-content,educational
1,aha,ott


In [59]:
df_main_app_category =  pd.merge(df_main_apps_explode, 
                                 category_list[['app_name', 'category']], 
                                 how='left', 
                                 left_on='app_lists', 
                                 right_on='app_name'
                              )
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,airtel thanks,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,airtel thanks,finance_transactions
1,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,dream11,fantasy_sports
2,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,dream11,fantasy_sports
3,62a1e8c317af9a37d451fd8a,0,AVAILABLE,zomato,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,zomato,delivery_food


#### Distribution check on categories

In [60]:
total_customers = df_main_app_category.customer_id.nunique()
total_customers

df_temp = df_main_app_category\
            .groupby(['category'])\
            .agg(app_list = ('app_lists', lambda x: list(set(x))),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,category,app_list,customers,%
0,tools,"[adobe lightroom, money lover, google drive, m...",1339692,96.02
1,social,"[moj, starmaker, josh, moj lite, smule, roposo...",1274670,91.36
2,finance_transactions,"[cred, money manager expense, mobikwik, google...",1193985,85.57
3,ott,"[voot, amazon prime video, jio cinema, disney+...",1125442,80.66
4,ecommerce,"[h&m, 1mg, shopsy, westside, flipkart, amazon ...",1033722,74.09
5,travel_ridehailing,"[yulu, tummoc, in drive, driveu, quickride, bl...",948144,67.95
6,streaming_music,"[amazon music, gaana, apple music, jiosaavn, h...",929732,66.64
7,delivery_food,"[starbucks india, zomato, kfc india, uber eats...",882621,63.26
8,office,"[google analytics, webex, dropbox, zoom, pocke...",747730,53.59
9,travel_bookings,"[makemytrip, irctc rail connect, abhibus, wher...",637678,45.70


#### Most Common used apps

In [61]:
df_temp = df_main_app_category\
            .groupby(['app_lists'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,app_lists,customers,%
0,google maps,1269615,91.00
1,youtube,1267489,90.84
2,gmail,1267404,90.84
3,whatsapp,1242209,89.03
4,google photos,1203205,86.24
...,...,...,...
414,fiitjee,4,0.00
415,cricplay,4,0.00
416,nishtha,4,0.00
417,meritnation,1,0.00


In [62]:
most_common_apps = ['google maps', 'gmail', 'youtube', 'whatsapp', 'clock', 'calculator',
                    'google calendar', 'google photos']

## All Apps as feature

#### Transform Categories

In [63]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,62a1e8c317af9a37d451fd8a,0,AVAILABLE,airtel thanks,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,airtel thanks,finance_transactions
1,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,dream11,fantasy_sports
2,62a1e8c317af9a37d451fd8a,0,AVAILABLE,dream11,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,dream11,fantasy_sports
3,62a1e8c317af9a37d451fd8a,0,AVAILABLE,zomato,45,9,4,6,39,5,4,PHH,12.0,HOOK,ONLY_AUTO,MALE,Unknown,health_and_personal,educational,Unknown,zomato,delivery_food


In [64]:
df_main_app_category[df_main_app_category['app_lists'].isna()].nunique()

customer_id                     0
age                             0
appo_presence                   0
app_lists                       0
fe_count                        0
rr_count                        0
net_count                       0
morning_fe                      0
evening_fe                      0
morning_rr                      0
evening_rr                      0
ltr_segment                     0
life_time_rides                 0
life_time_stage                 0
service_affinity                0
gender                          0
first_half_drop_primary         0
first_half_drop_secondary       0
second_half_pickup_primary      0
second_half_pickup_secondary    0
app_name                        0
category                        0
dtype: int64

In [65]:
df_customer_mapped_v1 = df_main_app_category[~df_main_app_category['app_lists'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_lists', lambda x: list(set(x))),
                             category = ('category', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_customer_mapped_v1.head(4)

,customer_id,age,app_list,category
0,5737c6adddbec2203f73316d,0,"[amazon prime video, disney+ hotstar, google d...","[ecommerce, driver_app, streaming_music, finan..."
1,5737c6aeddbec2203f733176,35,"[cred, zomato, instagram, microsoft 365, outlo...","[delivery_food, finance_transactions, social, ..."
2,5737c6b3ddbec2203f73319d,0,"[in drive, amazon prime video, disney+ hotstar...","[ecommerce, finance_transactions, social, trav..."
3,5737c6baddbec2203f7331d9,0,"[amazon prime video, disney+ hotstar, 99acres,...","[news, ecommerce, streaming_music, delivery_fo..."


In [66]:
unique_categories = set(category for sublist in df_customer_mapped_v1['app_list'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique apps:", len(unique_categories))
# sorted(unique_categories)

Count of unique apps: 411


In [67]:
df_customer_mapped_v1.head(2)

,customer_id,age,app_list,category
0,5737c6adddbec2203f73316d,0,"[amazon prime video, disney+ hotstar, google d...","[ecommerce, driver_app, streaming_music, finan..."
1,5737c6aeddbec2203f733176,35,"[cred, zomato, instagram, microsoft 365, outlo...","[delivery_food, finance_transactions, social, ..."


In [68]:
df_customer_mapped_v1.shape

(1394934, 4)

In [69]:
for category in unique_categories:
    df_customer_mapped_v1[category] = df_customer_mapped_v1['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v1.head(1)

,customer_id,age,app_list,category,moneycontrol,vision11,zoom,shemaroome livetv,udemy,pepperfry,fiitjee,google pay,github,myteam11,dream11,ajio,khan academy,line,aakash,instashop,trulymadly,course hero,fiton,lenskart,aha,swayam,tummoc,money lover,grofers delivery app,",",samsung galaxy wearable,freecharge,chennai metro rail,nps,aisle,todoist,zee5,flightaware,vero,ring,airtel thanks,microsoft teams,nishtha,angel broking,nba,geeks for geeks,abp live,myntra,makemytrip,dmrc momentum,...,mx takatak,wego,ms authenticator,wynk music,jio cinema,uber eats,google home,meritnation,uber driver app,hotstar,skymet weather,tunein radio,sharekhan,apple music,malayala manorama,splitwise,espncricinfo,zomato driver app,ola,nike training club,disney+ hotstar,espn,smule,amazon shopping,alt balaji,rapido captain,icicidirect: stocks,toppr,cisco,google chat,linkedin,viber,pizza hut india,woo,mx player,sun nxt,money manager expense,goodreads,dailyhunt,file commander,indigo,amazon music,driveu,sonyliv,club factory,happify,the souled store,booking.com,meesho,livspace
0,5737c6adddbec2203f73316d,0,"[amazon prime video, disney+ hotstar, google d...","[ecommerce, driver_app, streaming_music, finan...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


#### Bins definitions

In [71]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['Below-21', '21-25',  '26-30', '31-35', '36-45', 'Above-45']

df_customer_mapped_v1['age_group'] = pd.cut(df_customer_mapped_v1['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [72]:
main_list = ['customer_id', 'age', 'age_group' ,'category']
unique_column_list = main_list + unique_categories_list

In [73]:
df_customer_mapped_v1 = df_customer_mapped_v1[unique_column_list]
df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,moneycontrol,vision11,zoom,shemaroome livetv,udemy,pepperfry,fiitjee,google pay,github,myteam11,dream11,ajio,khan academy,line,aakash,instashop,trulymadly,course hero,fiton,lenskart,aha,swayam,tummoc,money lover,grofers delivery app,",",samsung galaxy wearable,freecharge,chennai metro rail,nps,aisle,todoist,zee5,flightaware,vero,ring,airtel thanks,microsoft teams,nishtha,angel broking,nba,geeks for geeks,abp live,myntra,makemytrip,dmrc momentum,...,mx takatak,wego,ms authenticator,wynk music,jio cinema,uber eats,google home,meritnation,uber driver app,hotstar,skymet weather,tunein radio,sharekhan,apple music,malayala manorama,splitwise,espncricinfo,zomato driver app,ola,nike training club,disney+ hotstar,espn,smule,amazon shopping,alt balaji,rapido captain,icicidirect: stocks,toppr,cisco,google chat,linkedin,viber,pizza hut india,woo,mx player,sun nxt,money manager expense,goodreads,dailyhunt,file commander,indigo,amazon music,driveu,sonyliv,club factory,happify,the souled store,booking.com,meesho,livspace
0,5737c6adddbec2203f73316d,0,NaN,"[ecommerce, driver_app, streaming_music, finan...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


In [74]:
# Adding required additional info by one-hot-encoding 

df_customer_mapped_v1 = pd.merge(df_customer_mapped_v1, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,moneycontrol,vision11,zoom,shemaroome livetv,udemy,pepperfry,fiitjee,google pay,github,myteam11,dream11,ajio,khan academy,line,aakash,instashop,trulymadly,course hero,fiton,lenskart,aha,swayam,tummoc,money lover,grofers delivery app,",",samsung galaxy wearable,freecharge,chennai metro rail,nps,aisle,todoist,zee5,flightaware,vero,ring,airtel thanks,microsoft teams,nishtha,angel broking,nba,geeks for geeks,abp live,myntra,makemytrip,dmrc momentum,...,service_affinity_ONLY_LINK,service_affinity_UNKNOWN,first_half_drop_primary_Unknown,first_half_drop_primary_educational,first_half_drop_primary_first_half_drop_primary_unknown,first_half_drop_primary_food,first_half_drop_primary_govt_amenity,first_half_drop_primary_health_and_personal,first_half_drop_primary_household_needs,first_half_drop_primary_leisure,first_half_drop_primary_office,first_half_drop_primary_place_of_worship,first_half_drop_primary_residential,first_half_drop_primary_transit_station,first_half_drop_secondary_Unknown,first_half_drop_secondary_educational,first_half_drop_secondary_first_half_drop_secondary_unknown,first_half_drop_secondary_food,first_half_drop_secondary_govt_amenity,first_half_drop_secondary_health_and_personal,first_half_drop_secondary_household_needs,first_half_drop_secondary_leisure,first_half_drop_secondary_office,first_half_drop_secondary_place_of_worship,first_half_drop_secondary_residential,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,5737c6adddbec2203f73316d,0,NaN,"[ecommerce, driver_app, streaming_music, finan...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [75]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v1.columns[df_customer_mapped_v1.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v1[columns_with_nan]
df_with_nan.isnull().sum()

age_group    979629
dtype: int64

In [76]:
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

## Stratified Sampling & Model

In [77]:
df_customer_mapped_v1_age_w = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_na = df_customer_mapped_v1[df_customer_mapped_v1['age_group'].isna()]

# Keep a copy of the columns to be retained
df_customer_mapped_v1_age_na_columns = df_customer_mapped_v1_age_na[['customer_id', 'category', 'age']].copy()

# Drop unnecessary columns for model training
df_customer_mapped_v1_age_w.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)
df_customer_mapped_v1_age_na.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)

In [78]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    classifiers = {}
    
    for target_class in target_classes:
        
        # Create a copy of df to avoid modifying the original dataframe
        df_temp = df.copy()
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Ensure all column names are strings
        df_combined.columns = df_combined.columns.astype(str)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        classifiers[target_class] = rf_model

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20] # Adjust 20 as needed
        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("Top Features:")
        for feature in top_features:
            print(feature)
        print("---------------------------------------")
        
    return classifiers

In [79]:
classifiers = evaluate_target_vs_rest(df_customer_mapped_v1_age_w, 'age_group')
classifiers

Results for class 21-25 vs Rest:
Accuracy :  71.44 %
Precision :  70.62 %
Recall :  72.85 %
F1 Score :  71.72 %
Top Features:
snapchat
messenger
google classroom
telegram
airtel thanks
wynk music
spotify
second_half_pickup_primary_second_half_pickup_primary_unknown
service_affinity_ONLY_AUTO
amazon prime video
bookmyshow
urban company
first_half_drop_primary_first_half_drop_primary_unknown
cred
service_affinity_ONLY_LINK
life_time_stage_HOOK
zomato
facebook
instagram
truecaller
---------------------------------------
Results for class 26-30 vs Rest:
Accuracy :  62.27 %
Precision :  62.64 %
Recall :  61.15 %
F1 Score :  61.89 %
Top Features:
airtel thanks
second_half_pickup_primary_second_half_pickup_primary_unknown
wynk music
first_half_drop_primary_first_half_drop_primary_unknown
snapchat
service_affinity_ONLY_AUTO
life_time_stage_HOOK
bookmyshow
service_affinity_ONLY_LINK
truecaller
spotify
amazon prime video
google news
messenger
telegram
paytm money
ola
netflix
uber
meesho
--------

{'21-25': RandomForestClassifier(random_state=42),
 '26-30': RandomForestClassifier(random_state=42),
 '31-35': RandomForestClassifier(random_state=42),
 '36-45': RandomForestClassifier(random_state=42),
 'Above-45': RandomForestClassifier(random_state=42),
 'Below-21': RandomForestClassifier(random_state=42)}

In [80]:
def predict_age_group(df, classifiers):
    predictions = pd.DataFrame(index=df.index)
    
    for target_class, classifier in classifiers.items():
        # Ensure the features used in prediction match the training data
        X = df[[col for col in df.columns if col != 'age_group']]
        predictions[f'{target_class}'] = classifier.predict_proba(X)[:, 1]
        
    binary_predictions = (predictions > 0.5).astype(int)
    
    return binary_predictions

In [81]:
# Use the models to predict the age group for df_customer_mapped_v1_age_na
predictions = predict_age_group(df_customer_mapped_v1_age_na, classifiers)
predictions

,21-25,26-30,31-35,36-45,Above-45,Below-21
0,1,1,0,0,0,0
2,0,0,1,1,0,0
3,0,0,1,1,0,0
4,0,0,1,1,1,0
5,0,1,1,1,0,0
...,...,...,...,...,...,...
1448586,0,1,0,0,0,0
1448587,1,0,0,0,0,1
1448588,0,0,1,1,1,1
1448589,1,0,0,0,0,1


In [82]:
predictions['total_sum'] = predictions.sum(axis=1)

# Merge the original columns with the prediction results
df_customer_mapped_v1_age_na_with_predictions = pd.concat([df_customer_mapped_v1_age_na_columns['customer_id'], predictions], axis=1)
df_customer_mapped_v1_age_na_with_predictions.head(2)

,customer_id,21-25,26-30,31-35,36-45,Above-45,Below-21,total_sum
0,5737c6adddbec2203f73316d,1,1,0,0,0,0,2
2,5737c6b3ddbec2203f73319d,0,0,1,1,0,0,2


In [83]:
df_customer_mapped_v1_age_na_with_predictions.total_sum.value_counts(normalize=True)

total_sum
2    0.458065
3    0.323066
1    0.119353
4    0.082213
0    0.010538
5    0.006537
6    0.000228
Name: proportion, dtype: float64

### Final Customer Age Class 

In [86]:
# Define a function to apply the logic
def age_weight_function(row):
    cols_with_one = [col for col in ['Below-21', '21-25', '26-30', '31-35', '36-45', 'Above-45'] if row[col] == 1]
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return cols_with_one[0]
    elif total_sum == 2:
        return cols_with_one[-1]
    elif total_sum == 3 or total_sum == 5:
        return cols_with_one[len(cols_with_one) // 2]
    elif total_sum == 4:
        return cols_with_one[2]
    elif total_sum == 6:
        return cols_with_one[4]
    else:
        return 'UNKNOWN'

def get_confidence_tag(row):
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return 'VERY_HIGH'
    elif total_sum == 2:
        return 'HIGH'
    elif total_sum == 3:
        return 'MEDIUM'
    elif total_sum in [4, 5, 6]:
        return 'LOW'
    else:
        return 'UNKNOWN'

In [87]:
df_customer_mapped_v1_age_na_with_predictions['age_group'] = df_customer_mapped_v1_age_na_with_predictions.apply(age_weight_function, axis=1)
df_customer_mapped_v1_age_na_with_predictions['confidence_tag'] = df_customer_mapped_v1_age_na_with_predictions.apply(get_confidence_tag, axis=1)

In [91]:
df_customer_mapped_v1_age_na_with_predictions

,customer_id,21-25,26-30,31-35,36-45,Above-45,Below-21,total_sum,age_group,confidence_tag
0,5737c6adddbec2203f73316d,1,1,0,0,0,0,2,26-30,HIGH
2,5737c6b3ddbec2203f73319d,0,0,1,1,0,0,2,36-45,HIGH
3,5737c6baddbec2203f7331d9,0,0,1,1,0,0,2,36-45,HIGH
4,5737c6d1ddbec2203f7332a8,0,0,1,1,1,0,3,36-45,MEDIUM
5,5737c6deddbec2203f733317,0,1,1,1,0,0,3,31-35,MEDIUM
...,...,...,...,...,...,...,...,...,...,...
1448586,668d809332b43c1042e5eb14,0,1,0,0,0,0,1,26-30,VERY_HIGH
1448587,668d80bd6db48cab82ba323c,1,0,0,0,0,1,2,21-25,HIGH
1448588,668d80ef6db48c1862ba38a9,0,0,1,1,1,1,4,36-45,LOW
1448589,668d80fe6db48ca302ba3adf,1,0,0,0,0,1,2,21-25,HIGH


In [89]:
df_customer_mapped_v1_age_na_with_predictions.age_group.value_counts(normalize=True)

age_group
21-25       0.319354
36-45       0.278521
26-30       0.169992
31-35       0.144651
Above-45    0.062259
Below-21    0.014686
UNKNOWN     0.010538
Name: proportion, dtype: float64

In [90]:
df_customer_mapped_v1_age_na_with_predictions.confidence_tag.value_counts(normalize=True)

confidence_tag
HIGH         0.458065
MEDIUM       0.323066
VERY_HIGH    0.119353
LOW          0.088978
UNKNOWN      0.010538
Name: proportion, dtype: float64

In [92]:
df_final1 = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()][['customer_id', 'age_group']]
df_final1['confidence_tag'] = 'VERY_HIGH'

df_final2 = df_customer_mapped_v1_age_na_with_predictions[['customer_id', 'age_group', 'confidence_tag']]

df_predicated_age_bins = pd.concat([df_final1,df_final2])
df_predicated_age_bins.shape

(1448591, 3)

In [93]:
df_predicated_age_bins.to_csv( local_extract + 'customer_predicted_age.csv', index=False)

In [94]:
df_predicated_age_bins.head(50)

,customer_id,age_group,confidence_tag
1,5737c6aeddbec2203f733176,36-45,VERY_HIGH
6,5737c6e3ddbec2203f733341,26-30,VERY_HIGH
7,5737c6f3ddbec2203f7333d1,26-30,VERY_HIGH
10,5737c712ddbec2203f7334d6,36-45,VERY_HIGH
13,5737c729ddbec2203f7335a5,31-35,VERY_HIGH
15,5737c730ddbec2203f7335de,36-45,VERY_HIGH
20,5737c749ddbec2203f7336b9,36-45,VERY_HIGH
23,5737c8e4ddbec220a7fa17ae,26-30,VERY_HIGH
25,5737ca80ddbec220de03eba4,26-30,VERY_HIGH
26,5737ca87ddbec220de03ebda,Above-45,VERY_HIGH
